# UC CosMx: 6K Panel Processing (Baseline)

QC filter, normalize, and AUCell annotate for 2 UC 6K samples (CD_B4, CD_B5).
Outputs used as 6K reference for imputation comparison (Fig 2).

In [ ]:
# ── Paths ──
DATA_DIR   = "/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k"
OUTPUT_DIR = DATA_DIR + "/Processed_merged_custom"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import anndata as ad
from matplotlib.backends.backend_pdf import PdfPages
import sys

In [ ]:
prefix = '/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/' 

In [ ]:
# prefix = directory that contains step1 reformatted sample folders e.g., '/share/fsmresfiles/UC/AtoMx/UST/cosmx/processed/'
# Renamed all sample folders to just sample name and response to keep things clean e.g., 'AGS1732860_AGS1918487_USTNR'
# this step reads in adata

for i in os.listdir(prefix):
    if 'Cos' in i: 
        print(i)
        sample_dir_write = prefix+i
        sample_dir = prefix+i+'/dir_formatted'
        fov_file = [item for item in os.listdir(sample_dir) if 'fov_positions_file' in item][0]
        fov_df = pd.read_csv(os.path.join(sample_dir, fov_file))
        if 'FOV' in fov_df.columns:
          print("Refactoring file to older format.")
          # Rename 'FOV' column to 'fov'
          fov_df.rename(columns={'FOV': 'fov'}, inplace=True)
          # have fov_file reference the new, formatted file and write it
          fov_file = os.path.join(sample_dir,'fov_positions_formatted.csv')
          fov_df.to_csv(fov_file, index=False)
        
        for i in os.listdir(sample_dir):
            if 'exprMat' in i:
                sample = i.split('_exprMat_file.csv.gz')[0]
        print(sample)
    
        adata = sq.read.nanostring(
            path=sample_dir,
            counts_file=sample+"_exprMat_file.csv.gz",
            meta_file=sample+"_metadata_file.csv.gz",
            fov_file="fov_positions_formatted.csv")
        print ('adata read in')
    
        to_drop = []
        for i in adata.obs.columns:
            if 'clust' in i:
                to_drop.append(i)
        print (to_drop)
    
        adata.obs = adata.obs.drop(to_drop,axis = 1)
        adata.write_h5ad(sample_dir_write+'raw_data.h5ad')


In [ ]:
adata1 = sc.read_h5ad(prefix+'Processed_CD_B4_custom/h5ad_obj/raw_data.h5ad')

In [ ]:
adata1

In [ ]:
adata1.var['features']=adata1.var.index.tolist()
adata1.obs[['orig.ident']]='CD_B4'
#adata.obs['patient']=['UC_P2' if int(i) <=31 else 'UC_P1' for i in adata.obs['fov'].tolist() ]
#adata.obs['condition']= 'Inflamed'
adata1.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata1.obs['fov'].tolist(), adata1.obs['cell_id'].tolist())]

In [ ]:
adata1.layers['raw']=adata1.X

In [ ]:
adata2 = sc.read_h5ad(prefix+'Processed_CD_B5_custom/h5ad_obj/raw_data.h5ad')

In [ ]:
adata2

In [ ]:
adata2.var['features']=adata2.var.index.tolist()
adata2.obs[['orig.ident']]='CD_B5'
#adata.obs['patient']=['UC_P2' if int(i) <=31 else 'UC_P1' for i in adata.obs['fov'].tolist() ]
#adata.obs['condition']= 'Inflamed'
adata2.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata2.obs['fov'].tolist(), adata2.obs['cell_id'].tolist())]

In [ ]:
adata2.layers['raw']=adata2.X

In [ ]:
slide=[]
slide1_list = range(0,15)
slide2_list = range(15,19)
slide3_list = range(19,31)
slide4_list = range(31,44)
slide5_list = range(44,52)

for i,j in zip(adata1.obs['orig.ident'].tolist(),adata1.obs['fov'].tolist()):
    j= int(j)
    if j in slide1_list:
        slide.append(i+'_'+'slide1')
    elif j in slide2_list:
        slide.append(i+'_'+'slide2')
    elif j in slide3_list:
        slide.append(i+'_'+'slide3')
    elif j in slide4_list:
        slide.append(i+'_'+'slide4')
    elif j in slide5_list:
        slide.append(i+'_'+'slide5')
    else:
        print('error')

In [ ]:
adata1.obs['slide']=slide

In [ ]:
slide=[]
slide1_list = range(1,11)
slide2_list = range(11,15)
slide3_list = range(15,34)
slide4_list = [34]
slide5_list = range(35,46)

for i,j in zip(adata2.obs['orig.ident'].tolist(),adata2.obs['fov'].tolist()):
    j= int(j)
    if j in slide1_list:
        slide.append(i+'_'+'slide1')
    elif j in slide2_list:
        slide.append(i+'_'+'slide2')
    elif j in slide3_list:
        slide.append(i+'_'+'slide3')
    elif j in slide4_list:
        slide.append(i+'_'+'slide4')
    elif j in slide5_list:
        slide.append(i+'_'+'slide5')
    else:
        print(j,'error')

In [ ]:
adata2.obs['slide']=slide

# qc

In [ ]:
adata1.var['neg_probe']=[True if 'Negative' in i else False for i in adata1.var.index.tolist()]
adata1.var['control_probe']=[True if 'SystemContro' in i else False for i in adata1.var.index.tolist()]
adata1.var

In [ ]:
adata2.var['neg_probe']=[True if 'Negative' in i else False for i in adata2.var.index.tolist()]
adata2.var['control_probe']=[True if 'SystemContro' in i else False for i in adata2.var.index.tolist()]
adata2.var

In [ ]:
adata1=adata1[:, ~adata1.var["neg_probe"]]
adata1=adata1[:, ~adata1.var["control_probe"]]

adata2=adata2[:, ~adata2.var["neg_probe"]]
adata2=adata2[:, ~adata2.var["control_probe"]]


In [ ]:
sc.pp.calculate_qc_metrics(
    adata1, inplace=True, log1p=True
)
sc.pp.calculate_qc_metrics(
    adata1_ls, inplace=True, log1p=True
)
sc.pp.calculate_qc_metrics(
    adata2, inplace=True, log1p=True
)
sc.pp.calculate_qc_metrics(
    adata2_ls, inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata1,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata1, min_counts=100)
sc.pp.filter_cells(adata1, min_genes=50)
sc.pp.filter_cells(adata1, max_counts=2500)
sc.pp.filter_cells(adata1, max_genes=1750)
adata1=adata1[adata1.obs['Area']<=30000]

In [ ]:
sc.pl.violin(
    adata1,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata2,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata2, min_counts=100)
sc.pp.filter_cells(adata2, min_genes=50)
sc.pp.filter_cells(adata2, max_counts=3000)
sc.pp.filter_cells(adata2, max_genes=2000)
adata2=adata2[adata2.obs['Area']<=25000]

In [ ]:
sc.pl.violin(
    adata2,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata_cosmx=sc.concat([adata1, adata2])

In [ ]:
sc.pp.normalize_total(adata_cosmx, inplace=True)
sc.pp.log1p(adata_cosmx)

In [ ]:
adata1.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_CD_B4_custom/h5ad_obj/qc.h5ad')
adata2.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_CD_B5_custom/h5ad_obj/qc.h5ad')

In [ ]:
adata_cosmx.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6K/Processed_merged_custom/h5ad_obj/qc_norm.h5ad')

In [ ]:
norm_log = adata_cosmx.X.copy()

In [ ]:
adata_cosmx.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_merged_custom/h5ad_obj/qc_norm_umap.h5ad')

In [ ]:
adata_cosmx1=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B4']
adata_cosmx2=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B5']

In [ ]:
adata_cosmx1.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_CD_B4_custom/h5ad_obj/qc.h5ad')
adata_cosmx2.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_CD_B5_custom/h5ad_obj/qc.h5ad')

In [ ]:
adata_cosmx_norm_nolog1p = adata_cosmx.copy()
adata_cosmx_norm_nolog1p.X = adata_cosmx_norm_nolog1p.layers['raw']

In [ ]:
sc.pp.normalize_total(adata_cosmx_norm_nolog1p, inplace=True)

In [ ]:
adata_cosmx_norm_nolog1p_count = pd.DataFrame(adata_cosmx_norm_nolog1p.X.toarray())


In [ ]:
adata_cosmx_norm_nolog1p_count.columns = adata_cosmx_norm_nolog1p.var.index.tolist()
adata_cosmx_norm_nolog1p_count.index=adata_cosmx_norm_nolog1p.obs['cell'].tolist()


In [ ]:
adata_cosmx_norm_nolog1p_count

In [ ]:
adata_cosmx_norm_nolog1p_count.to_csv('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_merged_custom/count_csv/norm_counts.csv')

# anno

In [ ]:
anno = pd.read_csv('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_merged_custom/anno/norm_counts_aucell.csv')

In [ ]:
anno

In [ ]:
mapper= pd.read_csv('/share/fsmresfiles/UC/scRNA-seq/merged_cd/cell_type_category_map.csv')

In [ ]:
# Step 1: Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Step 2: Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
anno['label_clean'] = anno['label'].str.rstrip()

In [ ]:
anno_map=anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')

In [ ]:
adata_cosmx.obs['cell_type'] = anno_map['cell type'].tolist()
adata_cosmx.obs['cell_type_short'] = anno_map['cell type short'].tolist()
adata_cosmx.obs['cell_category'] = anno_map['cell category'].tolist()
adata_cosmx.obs['cell_type_general'] = anno_map['cell type general'].tolist()

In [ ]:
original_cats = list(adata_cosmx.obs['cell_category'].astype('category').cat.categories)
reordered_cats = [cat for cat in original_cats if cat != 'Cycling'] + ['Cycling']

adata_cosmx.obs['cell_category'] = pd.Categorical(
    adata_cosmx.obs['cell_category'],
    categories=reordered_cats,
    ordered=True
)

In [ ]:
adata_cosmx1=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B4']
adata_cosmx2=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B5']

In [ ]:
adata_cosmx2_s3 = adata_cosmx2[adata_cosmx2.obs['slide']=='CD_B5_slide3']
adata_cosmx2_s3_ls = adata_cosmx_ls2[adata_cosmx_ls2.obs['slide']=='CD_B5_slide3']

In [ ]:
adata_cosmx1_s1 = adata_cosmx1[adata_cosmx1.obs['slide']=='CD_B4_slide1']
adata_cosmx1_s4 = adata_cosmx1[adata_cosmx1.obs['slide']=='CD_B4_slide4']

adata_cosmx1_s1_ls = adata_cosmx_ls1[adata_cosmx_ls1.obs['slide']=='CD_B4_slide1']
adata_cosmx1_s4_ls = adata_cosmx_ls1[adata_cosmx_ls1.obs['slide']=='CD_B4_slide4']

In [ ]:
# Create a color palette
coordinates = adata_cosmx1_s4_ls.obsm['spatial_fov']

category_column = 'cell_category'
categories = adata_cosmx1_s4_ls.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=2)  # Adjust `s` for marker size
plt.title("CD_B4 Slide 4 Spatial Plot for Cell Category No Custom")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()


In [ ]:
adata_cosmx.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_merged_custom/h5ad_obj/norm_anno.h5ad')


In [ ]:
patient =[]
condition = []

for i,j in adata_cosmx.obs.iterrows():
    if j['orig.ident'] == 'CD_B4':
        if int(j['fov']) <= 18:
            patient.append('CD_B4')
            condition.append('Inf')
        else:
            patient.append('UC_NI1')
            condition.append('Non-Inf')
    else:
        if int(j['fov']) <=14:
            patient.append('CD_B5')
            condition.append('Inf')
        else:
            patient.append('UC_NI2')
            condition.append('Healthy')

In [ ]:
adata_cosmx.obs['patient'] = patient
adata_cosmx.obs['condition'] = condition


In [ ]:
adata_cosmx1=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B4']
adata_cosmx2=adata_cosmx[adata_cosmx.obs['orig.ident']=='CD_B5']

In [ ]:
# Create a color palette
coordinates = adata_cosmx2.obsm['spatial_fov']

category_column = 'patient'
categories = adata_cosmx2.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
colors = categories.cat.codes.map(dict(enumerate(palette)))

# Plotting
plt.figure(figsize=(9, 10))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, s=0.5)  # Adjust `s` for marker size
plt.title("CD_B5 Spatial Plot for Patient Custom")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# Create a custom legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8) 
           for i in range(len(categories.cat.categories))]
plt.legend(handles, categories.cat.categories, title=category_column, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

In [ ]:
adata_cosmx.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_merged_custom/h5ad_obj/norm_anno.h5ad')
adata_cosmx_ls.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_merged/h5ad_obj/norm_anno.h5ad')

In [ ]:
adata_cosmx1.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_CD_B4_custom/h5ad_obj/anno.h5ad')
adata_cosmx2.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_CD_B5_custom/h5ad_obj/anno.h5ad')

In [ ]:
adata_cosmx_ls1.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_CD_B4/h5ad_obj/anno.h5ad')
adata_cosmx_ls2.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/6k/Processed_CD_B5/h5ad_obj/anno.h5ad')